### **Data**

In [1]:
import numpy as np

In [2]:
data = np.load('/content/drive/MyDrive/Lab/MOT metrics/nalain-lab/data/DPMtracker-data.npy', allow_pickle='TRUE').item()

In [3]:
keys = list(data.keys())
keys

['gt_ids',
 'tracker_ids',
 'gt_dets',
 'tracker_dets',
 'tracker_confidences',
 'similarity_scores',
 'num_tracker_dets',
 'num_gt_dets',
 'num_tracker_ids',
 'num_gt_ids',
 'num_timesteps',
 'seq']

In [ ]:
#frames  # 600 timesteps
for k in keys:
  print(f'{str(k)} --  {len(data[k]) if type(data[k]) is not int else "# "+str(data[k])}')

gt_ids --  600
tracker_ids --  600
gt_dets --  600
tracker_dets --  600
tracker_confidences --  600
similarity_scores --  600
num_tracker_dets --  # 7464
num_gt_dets --  # 18581
num_tracker_ids --  # 47
num_gt_ids --  # 62
num_timesteps --  # 600
seq --  12


In [ ]:
#seq
seq=data['seq']
seq

'MOT17-02-DPM'

In [ ]:
gt_ids=data['gt_ids']
print(gt_ids[0].shape)
gt_ids[-2:]

(22,)


[array([ 3,  4,  6,  8,  9, 15, 16, 17, 18, 19, 20, 21, 22, 23, 26, 31, 32,
        33, 34, 37, 41, 42, 44, 45, 53]),
 array([ 3,  4,  6,  8,  9, 15, 16, 17, 18, 19, 20, 21, 22, 23, 26, 31, 32,
        33, 34, 37, 41, 42, 44, 45, 53, 59, 60, 61])]

In [ ]:
gt_dets=data['gt_dets']
print(gt_dets[0].shape)
#gt_dets[-1:]

(22, 4)


In [ ]:
tracker_ids=data['tracker_ids']
print(tracker_ids[0].shape)
tracker_ids[-2:]

(8,)


[array([13, 20, 30, 31, 32, 33, 38, 42, 43, 44, 46]),
 array([13, 20, 30, 31, 32, 33, 38, 42, 43, 44, 46])]

In [ ]:
tracker_dets=data['tracker_dets']
print(tracker_dets[0].shape)
#tracker_dets[-1:]

(8, 4)


In [ ]:
tracker_confidences=data['tracker_confidences']
print(tracker_confidences[0].shape)
tracker_confidences[-1:]

#1 is given

(8,)


[array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])]

In [ ]:
#similarity scores
  
similarity_scores=data['similarity_scores']
print(similarity_scores[0].shape)
#similarity_scores[0].shape

float_formatter = "{:.1f}".format

for gtS in similarity_scores[0]:
  gtS=[float_formatter(x) for x in gtS]
  #print(gtS)

(22, 8)


(('John', 'Jenny'), ('Charles', 'Christy'), ('Mike', 'Monica'))

### **CLEAR MOT**

<pre>
CLR_TP         True Positives                                #
CLR_FN         False Negative                                #
CLR_FP         False Positives                               #
IDSW           Id switches                                   # 

MT             Mostly Tracked > 80%           0.8< count(tracks)/count(gt_det)
PT             20% - 80% tracked                           -----
ML             Mostly Lost < 20%                           -----

Frag                  currently tracked - previously not tracked (fragmentation)

MOTA           MOT Accuracy                (CLR_TP-CLR_FP-IDSW)/(CLR_TP+ CLR_FN)
MOTP           MOT Precision                    sum(matches::similarities)

MODA           MODetection Accuracy          (CLR_TP-CLR_FP)/(CLR_TP+ CLR_FN)

CLR_Re         Recall                             CLR_TP/(CLR_TP+ CLR_FN)       

CLR_Pr         Precision                          CLR_TP/(CLR_TP+ CLR_FP)


MTR            MT rate                           MT/ (MT+MP+ML)::matched
PTR            PT rate                           PT / (MT+MP+ML)::matched
MLR            ML rate                           ML Rate Ml/(MT+MP+ML)

sMOTA          S Mota                 (MOTP_sum-CLR_FP-IDSW)/(CLR_TP+ CLR_FN)
CLR_F1         F1 score          CLR_TP/ (CLR_TP+ 0.5*CLR_FN + 0.5*CLR_FP)

FP_per_frame   FP per frame           FP/ Timesteps
MOTAL          ----------             (CLR_TP-CLR_FP-log(IDSW))/(CLR_TP+ CLR_FN)


CLR_Frames     total number of frames --- timesteps


<b>The matching is done at detection level matching that optimizes the MOTA and MOTP score is chosen</b>

</pre>




**code**

In [ ]:
#data input data

In [ ]:
import numpy as np
from scipy.optimize import linear_sum_assignment

class CLEAR():
    """Class which implements the CLEAR metrics"""

    def __init__(self, config=None):
        main_integer_fields = ['CLR_TP', 'CLR_FN', 'CLR_FP', 'IDSW', 'MT', 'PT', 'ML', 'Frag']
        extra_integer_fields = ['CLR_Frames']
        self.integer_fields = main_integer_fields + extra_integer_fields
        main_float_fields = ['MOTA', 'MOTP', 'MODA', 'CLR_Re', 'CLR_Pr', 'MTR', 'PTR', 'MLR', 'sMOTA']
        extra_float_fields = ['CLR_F1', 'FP_per_frame', 'MOTAL', 'MOTP_sum']
        self.float_fields = main_float_fields + extra_float_fields
        self.fields = self.float_fields + self.integer_fields
        self.summed_fields = self.integer_fields + ['MOTP_sum']
        self.summary_fields = main_float_fields + main_integer_fields

        # Configuration options:
        self.threshold = float(0.5)


    
    def eval_sequence(self, data):
        """Calculates CLEAR metrics for one sequence"""
        # Initialise results
        res = {}
        for field in self.fields:
            res[field] = 0

        # Return result quickly if tracker or gt sequence is empty
        if data['num_tracker_dets'] == 0:
            res['CLR_FN'] = data['num_gt_dets']
            res['ML'] = data['num_gt_ids']
            res['MLR'] = 1.0
            return res
        if data['num_gt_dets'] == 0:
            res['CLR_FP'] = data['num_tracker_dets']
            res['MLR'] = 1.0
            return res

        # Variables counting global association
        num_gt_ids = data['num_gt_ids']
        gt_id_count = np.zeros(num_gt_ids)  # For MT/ML/PT
        gt_matched_count = np.zeros(num_gt_ids)  # For MT/ML/PT
        gt_frag_count = np.zeros(num_gt_ids)  # For Frag

        # Note that IDSWs are counted based on the last time each gt_id was present (any number of frames previously),
        # but are only used in matching to continue current tracks based on the gt_id in the single previous timestep.
        prev_tracker_id = np.nan * np.zeros(num_gt_ids)  # For scoring IDSW
        prev_timestep_tracker_id = np.nan * np.zeros(num_gt_ids)  # For matching IDSW

        # Calculate scores for each timestep
        for t, (gt_ids_t, tracker_ids_t) in enumerate(zip(data['gt_ids'], data['tracker_ids'])):
            # Deal with the case that there are no gt_det/tracker_det in a timestep.
            if len(gt_ids_t) == 0:
                res['CLR_FP'] += len(tracker_ids_t)
                continue
            if len(tracker_ids_t) == 0:
                res['CLR_FN'] += len(gt_ids_t)
                gt_id_count[gt_ids_t] += 1
                continue

            # Calc score matrix to first minimise IDSWs from previous frame, and then maximise MOTP secondarily
            similarity = data['similarity_scores'][t]
            score_mat = (tracker_ids_t[np.newaxis, :] == prev_timestep_tracker_id[gt_ids_t[:, np.newaxis]])
            score_mat = 1000 * score_mat + similarity
            score_mat[similarity < self.threshold - np.finfo('float').eps] = 0

            # Hungarian algorithm to find best matches
            match_rows, match_cols = linear_sum_assignment(-score_mat)
            actually_matched_mask = score_mat[match_rows, match_cols] > 0 + np.finfo('float').eps
            match_rows = match_rows[actually_matched_mask]
            match_cols = match_cols[actually_matched_mask]

            matched_gt_ids = gt_ids_t[match_rows]
            matched_tracker_ids = tracker_ids_t[match_cols]

            # Calc IDSW for MOTA
            prev_matched_tracker_ids = prev_tracker_id[matched_gt_ids]
            is_idsw = (np.logical_not(np.isnan(prev_matched_tracker_ids))) & (
                np.not_equal(matched_tracker_ids, prev_matched_tracker_ids))
            res['IDSW'] += np.sum(is_idsw)

            # Update counters for MT/ML/PT/Frag and record for IDSW/Frag for next timestep
            gt_id_count[gt_ids_t] += 1
            gt_matched_count[matched_gt_ids] += 1
            not_previously_tracked = np.isnan(prev_timestep_tracker_id)
            prev_tracker_id[matched_gt_ids] = matched_tracker_ids
            prev_timestep_tracker_id[:] = np.nan
            prev_timestep_tracker_id[matched_gt_ids] = matched_tracker_ids
            currently_tracked = np.logical_not(np.isnan(prev_timestep_tracker_id))
            gt_frag_count += np.logical_and(not_previously_tracked, currently_tracked)

            # Calculate and accumulate basic statistics
            num_matches = len(matched_gt_ids)
            res['CLR_TP'] += num_matches
            res['CLR_FN'] += len(gt_ids_t) - num_matches
            res['CLR_FP'] += len(tracker_ids_t) - num_matches
            if num_matches > 0:
                res['MOTP_sum'] += sum(similarity[match_rows, match_cols])

        # Calculate MT/ML/PT/Frag/MOTP
            tracked_ratio = gt_matched_count[gt_id_count > 0] / gt_id_count[gt_id_count > 0]
        res['MT'] = np.sum(np.greater(tracked_ratio, 0.8))
        res['PT'] = np.sum(np.greater_equal(tracked_ratio, 0.2)) - res['MT']
        res['ML'] = num_gt_ids - res['MT'] - res['PT']
        res['Frag'] = np.sum(np.subtract(gt_frag_count[gt_frag_count > 0], 1))
        res['MOTP'] = res['MOTP_sum'] / np.maximum(1.0, res['CLR_TP'])

        res['CLR_Frames'] = data['num_timesteps']

        # Calculate final CLEAR scores
        res = self._compute_final_fields(res)
        return res



    @staticmethod
    def _compute_final_fields(res):
        """Calculate sub-metric ('field') values which only depend on other sub-metric values.
        This function is used both for both per-sequence calculation, and in combining values across sequences.
        """
        num_gt_ids = res['MT'] + res['ML'] + res['PT']
        res['MTR'] = res['MT'] / np.maximum(1.0, num_gt_ids)
        res['MLR'] = res['ML'] / np.maximum(1.0, num_gt_ids)
        res['PTR'] = res['PT'] / np.maximum(1.0, num_gt_ids)
        res['CLR_Re'] = res['CLR_TP'] / np.maximum(1.0, res['CLR_TP'] + res['CLR_FN'])
        res['CLR_Pr'] = res['CLR_TP'] / np.maximum(1.0, res['CLR_TP'] + res['CLR_FP'])
        res['MODA'] = (res['CLR_TP'] - res['CLR_FP']) / np.maximum(1.0, res['CLR_TP'] + res['CLR_FN'])
        res['MOTA'] = (res['CLR_TP'] - res['CLR_FP'] - res['IDSW']) / np.maximum(1.0, res['CLR_TP'] + res['CLR_FN'])
        res['MOTP'] = res['MOTP_sum'] / np.maximum(1.0, res['CLR_TP'])
        res['sMOTA'] = (res['MOTP_sum'] - res['CLR_FP'] - res['IDSW']) / np.maximum(1.0, res['CLR_TP'] + res['CLR_FN'])

        res['CLR_F1'] = res['CLR_TP'] / np.maximum(1.0, res['CLR_TP'] + 0.5*res['CLR_FN'] + 0.5*res['CLR_FP'])
        res['FP_per_frame'] = res['CLR_FP'] / np.maximum(1.0, res['CLR_Frames'])
        safe_log_idsw = np.log10(res['IDSW']) if res['IDSW'] > 0 else res['IDSW']
        res['MOTAL'] = (res['CLR_TP'] - res['CLR_FP'] - safe_log_idsw) / np.maximum(1.0, res['CLR_TP'] + res['CLR_FN'])

        
        return res

In [ ]:
#data
data=data

metrics=CLEAR()
results=metrics.eval_sequence(data)

In [ ]:
results

{'MOTA': 0.3903987944674668,
 'MOTP': 0.9175379091707903,
 'MODA': 0.39136752596738605,
 'CLR_Re': 0.3965340939669555,
 'CLR_Pr': 0.9871382636655949,
 'MTR': 0.1774193548387097,
 'PTR': 0.3548387096774194,
 'MLR': 0.46774193548387094,
 'sMOTA': 0.3576997639938853,
 'CLR_F1': 0.565789978882703,
 'FP_per_frame': 0.16,
 'MOTAL': 0.39129996918868176,
 'MOTP_sum': 6760.419314770383,
 'CLR_TP': 7368,
 'CLR_FN': 11213,
 'CLR_FP': 96,
 'IDSW': 18,
 'MT': 11,
 'PT': 22,
 'ML': 29,
 'Frag': 19.0,
 'CLR_Frames': 600}

### **Identity**

<pre> 


IDTP                    Identity tp        ( matched   trDt)
IDFN                    Identity fn        ( unmatched gtDet)
IDFP                    Identity fp        ( unmatched trDets)

IDR                     Id Recall            IDTP/(IDTP+IDFN)
IDP                     Id Precision         IDTP/(IDTP+IDFP)

IDF1                    Id F1Score         IDTP/(IDTP+0.5 IDFN +0.5 IDFP)


<b>The matching is done at association level matching that optimizes the IDF1 score is chosen (i.e reducing FP and FN)</b>

</pre>

**code**

In [ ]:
import numpy as np
from scipy.optimize import linear_sum_assignment


class Identity():
    """Class which implements the ID metrics"""


    def __init__(self, config=None):
        self.integer_fields = ['IDTP', 'IDFN', 'IDFP']
        self.float_fields = ['IDF1', 'IDR', 'IDP']
        self.fields = self.float_fields + self.integer_fields
        self.summary_fields = self.fields

        # Configuration options:
        self.threshold = float(0.5)

    def eval_sequence(self, data):
        """Calculates ID metrics for one sequence"""
        # Initialise results
        res = {}
        for field in self.fields:
            res[field] = 0

        # Return result quickly if tracker or gt sequence is empty
        if data['num_tracker_dets'] == 0:
            res['IDFN'] = data['num_gt_dets']
            return res
        if data['num_gt_dets'] == 0:
            res['IDFP'] = data['num_tracker_dets']
            return res

        # Variables counting global association
        potential_matches_count = np.zeros((data['num_gt_ids'], data['num_tracker_ids']))
        gt_id_count = np.zeros(data['num_gt_ids'])
        tracker_id_count = np.zeros(data['num_tracker_ids'])

        # First loop through each timestep and accumulate global track information.
        for t, (gt_ids_t, tracker_ids_t) in enumerate(zip(data['gt_ids'], data['tracker_ids'])):
            # Count the potential matches between ids in each timestep
            matches_mask = np.greater_equal(data['similarity_scores'][t], self.threshold)
            match_idx_gt, match_idx_tracker = np.nonzero(matches_mask)
            potential_matches_count[gt_ids_t[match_idx_gt], tracker_ids_t[match_idx_tracker]] += 1

            # Calculate the total number of dets for each gt_id and tracker_id.
            gt_id_count[gt_ids_t] += 1
            tracker_id_count[tracker_ids_t] += 1

        # Calculate optimal assignment cost matrix for ID metrics
        num_gt_ids = data['num_gt_ids']
        num_tracker_ids = data['num_tracker_ids']
        fp_mat = np.zeros((num_gt_ids + num_tracker_ids, num_gt_ids + num_tracker_ids))
        fn_mat = np.zeros((num_gt_ids + num_tracker_ids, num_gt_ids + num_tracker_ids))
        fp_mat[num_gt_ids:, :num_tracker_ids] = 1e10
        fn_mat[:num_gt_ids, num_tracker_ids:] = 1e10
        for gt_id in range(num_gt_ids):
            fn_mat[gt_id, :num_tracker_ids] = gt_id_count[gt_id]
            fn_mat[gt_id, num_tracker_ids + gt_id] = gt_id_count[gt_id]
        for tracker_id in range(num_tracker_ids):
            fp_mat[:num_gt_ids, tracker_id] = tracker_id_count[tracker_id]
            fp_mat[tracker_id + num_gt_ids, tracker_id] = tracker_id_count[tracker_id]
        fn_mat[:num_gt_ids, :num_tracker_ids] -= potential_matches_count
        fp_mat[:num_gt_ids, :num_tracker_ids] -= potential_matches_count

        # Hungarian algorithm
        match_rows, match_cols = linear_sum_assignment(fn_mat + fp_mat)

        # Accumulate basic statistics
        res['IDFN'] = fn_mat[match_rows, match_cols].sum().astype(np.int)
        res['IDFP'] = fp_mat[match_rows, match_cols].sum().astype(np.int)
        res['IDTP'] = (gt_id_count.sum() - res['IDFN']).astype(np.int)

        # Calculate final ID scores
        res = self._compute_final_fields(res)
        return res

    @staticmethod
    def _compute_final_fields(res):
        """Calculate sub-metric ('field') values which only depend on other sub-metric values.
        This function is used both for both per-sequence calculation, and in combining values across sequences.
        """
        res['IDR'] = res['IDTP'] / np.maximum(1.0, res['IDTP'] + res['IDFN'])
        res['IDP'] = res['IDTP'] / np.maximum(1.0, res['IDTP'] + res['IDFP'])
        res['IDF1'] = res['IDTP'] / np.maximum(1.0, res['IDTP'] + 0.5 * res['IDFP'] + 0.5 * res['IDFN'])
        return res


In [ ]:
#data
data=data

metrics=Identity()
results=metrics.eval_sequence(data)

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:69: DeprecationWarning: `np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:70: DeprecationWarning: `np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
Deprecat

In [ ]:
results

{'IDF1': 0.4860817815319639,
 'IDR': 0.3406705774716108,
 'IDP': 0.8480707395498392,
 'IDTP': 6330,
 'IDFN': 12251,
 'IDFP': 1134}

### **HOTA**


<pre>


HOTA_TP               tp
HOTA_FN               fn
HOTA_FP               fp


HOTA                  High Order Tracking Accuracy
DetA                  Det Accuracy
AssA                  Association accuracy

DetRe                 Detection Recall
DetPr                 Detection Precision
AssRe                 Association Recall
AssPr                 Association Precision
LocA                  Localization Accuracy


RHOTA                 ----
HOTA(0)               ----
LocA(0)               ----
HOTALocA(0)           ----

<b>Considers both detections, association and localization</b>
</pre>

**Code**

In [4]:
import numpy as np
from scipy.optimize import linear_sum_assignment


class HOTA():
    """Class which implements the HOTA metrics.
    See: https://link.springer.com/article/10.1007/s11263-020-01375-2
    """

    def __init__(self, config=None):
        self.plottable = True
        self.array_labels = np.arange(0.05, 0.99, 0.05)
        self.integer_array_fields = ['HOTA_TP', 'HOTA_FN', 'HOTA_FP']
        self.float_array_fields = ['HOTA', 'DetA', 'AssA', 'DetRe', 'DetPr', 'AssRe', 'AssPr', 'LocA', 'RHOTA']
        self.float_fields = ['HOTA(0)', 'LocA(0)', 'HOTALocA(0)']
        self.fields = self.float_array_fields + self.integer_array_fields + self.float_fields
        self.summary_fields = self.float_array_fields + self.float_fields

   
    def eval_sequence(self, data):
        """Calculates the HOTA metrics for one sequence"""

        # Initialise results
        res = {}
        for field in self.float_array_fields + self.integer_array_fields:
            res[field] = np.zeros((len(self.array_labels)), dtype=np.float)
        for field in self.float_fields:
            res[field] = 0

        # Return result quickly if tracker or gt sequence is empty
        if data['num_tracker_dets'] == 0:
            res['HOTA_FN'] = data['num_gt_dets'] * np.ones((len(self.array_labels)), dtype=np.float)
            res['LocA'] = np.ones((len(self.array_labels)), dtype=np.float)
            res['LocA(0)'] = 1.0
            return res
        if data['num_gt_dets'] == 0:
            res['HOTA_FP'] = data['num_tracker_dets'] * np.ones((len(self.array_labels)), dtype=np.float)
            res['LocA'] = np.ones((len(self.array_labels)), dtype=np.float)
            res['LocA(0)'] = 1.0
            return res

        # Variables counting global association
        potential_matches_count = np.zeros((data['num_gt_ids'], data['num_tracker_ids']))
        gt_id_count = np.zeros((data['num_gt_ids'], 1))
        tracker_id_count = np.zeros((1, data['num_tracker_ids']))

        # First loop through each timestep and accumulate global track information.
        for t, (gt_ids_t, tracker_ids_t) in enumerate(zip(data['gt_ids'], data['tracker_ids'])):
            # Count the potential matches between ids in each timestep
            # These are normalised, weighted by the match similarity.
            similarity = data['similarity_scores'][t]
            sim_iou_denom = similarity.sum(0)[np.newaxis, :] + similarity.sum(1)[:, np.newaxis] - similarity
            sim_iou = np.zeros_like(similarity)
            sim_iou_mask = sim_iou_denom > 0 + np.finfo('float').eps
            sim_iou[sim_iou_mask] = similarity[sim_iou_mask] / sim_iou_denom[sim_iou_mask]
            potential_matches_count[gt_ids_t[:, np.newaxis], tracker_ids_t[np.newaxis, :]] += sim_iou

            # Calculate the total number of dets for each gt_id and tracker_id.
            gt_id_count[gt_ids_t] += 1
            tracker_id_count[0, tracker_ids_t] += 1

        # Calculate overall jaccard alignment score (before unique matching) between IDs
        global_alignment_score = potential_matches_count / (gt_id_count + tracker_id_count - potential_matches_count)
        matches_counts = [np.zeros_like(potential_matches_count) for _ in self.array_labels]

        # Calculate scores for each timestep
        for t, (gt_ids_t, tracker_ids_t) in enumerate(zip(data['gt_ids'], data['tracker_ids'])):
            # Deal with the case that there are no gt_det/tracker_det in a timestep.
            if len(gt_ids_t) == 0:
                for a, alpha in enumerate(self.array_labels):
                    res['HOTA_FP'][a] += len(tracker_ids_t)
                continue
            if len(tracker_ids_t) == 0:
                for a, alpha in enumerate(self.array_labels):
                    res['HOTA_FN'][a] += len(gt_ids_t)
                continue

            # Get matching scores between pairs of dets for optimizing HOTA
            similarity = data['similarity_scores'][t]
            score_mat = global_alignment_score[gt_ids_t[:, np.newaxis], tracker_ids_t[np.newaxis, :]] * similarity

            # Hungarian algorithm to find best matches
            match_rows, match_cols = linear_sum_assignment(-score_mat)

            # Calculate and accumulate basic statistics
            for a, alpha in enumerate(self.array_labels):
                actually_matched_mask = similarity[match_rows, match_cols] >= alpha - np.finfo('float').eps
                alpha_match_rows = match_rows[actually_matched_mask]
                alpha_match_cols = match_cols[actually_matched_mask]
                num_matches = len(alpha_match_rows)
                res['HOTA_TP'][a] += num_matches
                res['HOTA_FN'][a] += len(gt_ids_t) - num_matches
                res['HOTA_FP'][a] += len(tracker_ids_t) - num_matches
                if num_matches > 0:
                    res['LocA'][a] += sum(similarity[alpha_match_rows, alpha_match_cols])
                    matches_counts[a][gt_ids_t[alpha_match_rows], tracker_ids_t[alpha_match_cols]] += 1

        # Calculate association scores (AssA, AssRe, AssPr) for the alpha value.
        # First calculate scores per gt_id/tracker_id combo and then average over the number of detections.
        for a, alpha in enumerate(self.array_labels):
            matches_count = matches_counts[a]
            ass_a = matches_count / np.maximum(1, gt_id_count + tracker_id_count - matches_count)
            res['AssA'][a] = np.sum(matches_count * ass_a) / np.maximum(1, res['HOTA_TP'][a])
            ass_re = matches_count / np.maximum(1, gt_id_count)
            res['AssRe'][a] = np.sum(matches_count * ass_re) / np.maximum(1, res['HOTA_TP'][a])
            ass_pr = matches_count / np.maximum(1, tracker_id_count)
            res['AssPr'][a] = np.sum(matches_count * ass_pr) / np.maximum(1, res['HOTA_TP'][a])

        # Calculate final scores
        res['LocA'] = np.maximum(1e-10, res['LocA']) / np.maximum(1e-10, res['HOTA_TP'])
        res = self._compute_final_fields(res)
        return res



    @staticmethod
    def _compute_final_fields(res):
        """Calculate sub-metric ('field') values which only depend on other sub-metric values.
        This function is used both for both per-sequence calculation, and in combining values across sequences.
        """
        res['DetRe'] = res['HOTA_TP'] / np.maximum(1, res['HOTA_TP'] + res['HOTA_FN'])
        res['DetPr'] = res['HOTA_TP'] / np.maximum(1, res['HOTA_TP'] + res['HOTA_FP'])
        res['DetA'] = res['HOTA_TP'] / np.maximum(1, res['HOTA_TP'] + res['HOTA_FN'] + res['HOTA_FP'])
        res['HOTA'] = np.sqrt(res['DetA'] * res['AssA'])
        res['RHOTA'] = np.sqrt(res['DetRe'] * res['AssA'])

        res['HOTA(0)'] = res['HOTA'][0]
        res['LocA(0)'] = res['LocA'][0]
        res['HOTALocA(0)'] = res['HOTA(0)']*res['LocA(0)']
        return res

In [5]:
#data
data=data

metrics=HOTA()
results=metrics.eval_sequence(data)

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:26: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations


In [52]:
#these are the similarity thresholds used in computing HOTAalpha
thresholds=np.arange(0.05, 0.99, 0.05)
print(results['HOTA'].shape)
MKeys=np.array([k  for k in results.keys() if type(results[k]) is np.ndarray])

#results


(19,)


In [57]:
hota=(1/19) * np.sum(results['HOTA'])
print(hota)
print(results["HOTA(0)"])

0.43339253871587247
0.4705538054757147


### **Count**

<pre>       
<b>Counts accumulated over all frames(timesteps) </b>


Dets                            tracker detections
GT_Dets                         groundtruth detections
IDs                             tracker Ids
GT_IDs                          groundtruth Ids
Frames                          Total number of frames
</pre>

**Code**

In [ ]:
import numpy as np
from scipy.optimize import linear_sum_assignment


class Count():
    """Class which simply counts the number of tracker and gt detections and ids."""
    def __init__(self, config=None):
        self.integer_fields = ['Dets', 'GT_Dets', 'IDs', 'GT_IDs']
        self.fields = self.integer_fields
        self.summary_fields = self.fields

    def eval_sequence(self, data):
        """Returns counts for one sequence"""
        # Get results
        res = {'Dets': data['num_tracker_dets'],
               'GT_Dets': data['num_gt_dets'],
               'IDs': data['num_tracker_ids'],
               'GT_IDs': data['num_gt_ids'],
               'Frames': data['num_timesteps']}
        return res


In [ ]:
#data
data=data

metrics=Count()
results=metrics.eval_sequence(data)

In [ ]:
results

{'Dets': 7464, 'GT_Dets': 18581, 'IDs': 47, 'GT_IDs': 62, 'Frames': 600}